In [6]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# ----------------------------------------------------
# 1. Setup & Data Loading
# ----------------------------------------------------
data_dir = "/content"

print("Loading data...")
train_ridership = pd.read_csv(f"{data_dir}/Train_Ridership_2022_to_2025H1.csv")
shock_ridership = pd.read_csv(f"{data_dir}/Shock_Ridership_2025_Q3.csv")
bus_routes = pd.read_csv(f"{data_dir}/Bus_Routes.csv")

# Merge route type into historical data
train_ridership = train_ridership.merge(bus_routes[['Route_ID', 'Route_Type', 'Route_Code']], on='Route_ID', how='left')

# Helper: Aggregate Daily Route Passengers
def get_daily_route_pax(df):
    df['Date'] = pd.to_datetime(df['Date'])
    df['Total_Pax'] = df['Boarding_Count'] + df['Alighting_Count']
    return df.groupby(['Date', 'Route_Code', 'Route_Type'])['Total_Pax'].sum().reset_index()

train_agg = get_daily_route_pax(train_ridership)
shock_agg = get_daily_route_pax(shock_ridership)


# ----------------------------------------------------
# A. Detect Structural Break (Shift Classification)
# ----------------------------------------------------
print("Quantifying Stage 1 vs Q3 Actual Deviations...")

# Baseline: 2025 H1 Performance
h1_2025 = train_agg[train_agg['Date'].dt.year == 2025].copy()
route_baseline = h1_2025.groupby('Route_Code').agg(
    H1_Mean=('Total_Pax', 'mean'),
    H1_Std=('Total_Pax', 'std')
).reset_index()

# Shock: 2025 Q3 Performance
route_shock = shock_agg.groupby('Route_Code').agg(
    Q3_Mean=('Total_Pax', 'mean'),
    Q3_Std=('Total_Pax', 'std')
).reset_index()

# Merge and calculate divergence magnitudes
shift_analysis = route_baseline.merge(route_shock, on='Route_Code')
shift_analysis = shift_analysis.merge(bus_routes[['Route_Code', 'Route_Type']].drop_duplicates(), on='Route_Code')

shift_analysis['Level_Shift_PCT'] = ((shift_analysis['Q3_Mean'] - shift_analysis['H1_Mean']) / shift_analysis['H1_Mean']) * 100
shift_analysis['Volatility_Shift_PCT'] = ((shift_analysis['Q3_Std'] - shift_analysis['H1_Std']) / shift_analysis['H1_Std']) * 100

# Step A: Shift Classification Logic
conditions = [
    (shift_analysis['Level_Shift_PCT'] > 15) & (shift_analysis['Volatility_Shift_PCT'] > 20),
    (shift_analysis['Level_Shift_PCT'] > 15),
    (shift_analysis['Level_Shift_PCT'] < -5),
    (shift_analysis['Volatility_Shift_PCT'] > 25)
]
choices = [
    'Combined Level & Volatility Shift',
    'Level Shift (Demand Surge)',
    'Negative Level Shift (Contraction/Substitution)',
    'Volatility Shift (Erratic Demand)'
]
shift_analysis['Shift_Classification'] = np.select(conditions, choices, default='Congestion-mediated Shift (Stable)')


# ----------------------------------------------------
# B. Recalibrate Forecast for Q4 (Oct 1 - Dec 31, 2025)
# ----------------------------------------------------
print("Recalibrating Q4 Forecasts based on Shift Mechanisms...")

def recalibrate_q4(row):
    if row['Route_Type'] == 'Express':
        # Regime Change: Sustained High (Permanent Shift)
        return row['Q3_Mean'] * 1.05
    elif row['Route_Type'] == 'Feeder':
        # Regime Change: Permanent Contraction due to Metro Phase 2 Substitution
        return row['Q3_Mean'] * 0.90
    else:
        # Temporary Shock / Elasticity adjustment: Decay logic between H1 normal and Shock peak
        return (row['H1_Mean'] * 0.4) + (row['Q3_Mean'] * 0.6)

shift_analysis['Recalibrated_Q4_Forecast'] = shift_analysis.apply(recalibrate_q4, axis=1)
shift_analysis['Forecast_Logic'] = np.where(shift_analysis['Route_Type'] == 'Express', 'Regime Change (Sustained)',
                                    np.where(shift_analysis['Route_Type'] == 'Feeder', 'Metro Substitution (Permanent Decline)',
                                    'Elasticity Adjustment (Partial Decay)'))


# ----------------------------------------------------
# C. Operational Risk Reassessment
# ----------------------------------------------------
print("Reassessing Operational Constraints & Risks...")

shift_analysis['Operational_Risk'] = 'Stable Operation'
# High demand triggers Overload constraints (Proxy threshold: 450/daily is very high for this sample dataset)
shift_analysis.loc[shift_analysis['Recalibrated_Q4_Forecast'] > 450, 'Operational_Risk'] = 'Overload Risk (Requires Expansion)'
# If Recalibrated Forecast is lower than historical H1 baseline, Feeder viability is compromised
shift_analysis.loc[shift_analysis['Recalibrated_Q4_Forecast'] < shift_analysis['H1_Mean'], 'Operational_Risk'] = 'Underutilized (Feeder Viability at Risk)'


# ----------------------------------------------------
# D. Plotly Chart Visualizations (Using fig.show())
# ----------------------------------------------------
print("Rendering Final Stage 2 Outputs...\n")

# --- PLOT 1: Structural Break Diagnostic Map ---
fig1 = px.scatter(
    shift_analysis,
    x='Level_Shift_PCT', y='Volatility_Shift_PCT',
    color='Shift_Classification', size='Q3_Mean', text='Route_Code',
    title='Diagnostic Matrix: Categorizing Metro Phase 2 Structural Breaks',
    labels={'Level_Shift_PCT': 'Level Shift Magnitude (+/- %)', 'Volatility_Shift_PCT': 'Volatility Shift (+/- % StdDev)'},
    hover_data=['Route_Type']
)
fig1.add_hline(y=0, line_width=1, line_dash="solid", line_color="black")
fig1.add_vline(x=0, line_width=1, line_dash="solid", line_color="black")
fig1.update_traces(textposition='top center')
fig1.update_layout(template='plotly_white')
fig1.show()


# --- PLOT 2: Risk Flagging & Forecast Output ---
fig2 = px.bar(
    shift_analysis.sort_values('Recalibrated_Q4_Forecast', ascending=False),
    x='Route_Code', y=['H1_Mean', 'Q3_Mean', 'Recalibrated_Q4_Forecast'],
    title='Recalibrated Q4 Demands vs Historical Baselines (H1 & Q3)',
    labels={'value': 'Average Daily Passengers', 'variable': 'Modeling Period', 'Route_Code': 'Transport Corridor'},
    barmode='group'
)
# Visually annotate risks with background shapes
for idx, row in shift_analysis.iterrows():
    if "Overload" in row['Operational_Risk']:
        color = "red"
    elif "Underutilized" in row['Operational_Risk']:
        color = "orange"
    else:
        color = "green"
    fig2.add_shape(
        type="rect",
        x0=row['Route_Code'], x1=row['Route_Code'], y0=0, y1=row['Recalibrated_Q4_Forecast'],
        line=dict(color=color, width=4)
    )
fig2.update_layout(template='plotly_white', hovermode='x unified')
fig2.show()


# --- PLOT 3: Fleet Reallocation & Redistribution Strategy (Waterfall) ---
# Summing the net shift percentages by Route Type to dictate where buses move
reallocation_net = shift_analysis.groupby('Route_Type')['Level_Shift_PCT'].mean().values
reallocation_labels = [f"{val:.1f}%" for val in reallocation_net]
route_classes = shift_analysis['Route_Type'].unique()

fig3 = go.Figure(go.Waterfall(
    name="Routes", orientation="v",
    measure=["relative", "relative", "relative", "relative"],
    x=route_classes,
    textposition="outside",
    y=reallocation_net,
    text=reallocation_labels,
    decreasing={"marker":{"color":"#ff6b6b"}},
    increasing={"marker":{"color":"#1dd1a1"}},
    totals={"marker":{"color":"#54a0ff"}}
))
fig3.update_layout(
    title='Fleet Redeployment Pipeline: Net System Passenger Redistribution',
    xaxis_title='Corridor Archetypes',
    yaxis_title='Average Shift from Baseline (%)',
    template='plotly_white'
)
fig3.show()

# --- D. Final Text Strategy Output ---
print("STRATEGIC SUMMARY:")
print("1. Feeder rationalization is mandatory. They suffered Negative Level Shifts indicating Metro-substitution.")
print("2. Express Corridors (X66, X28, X11) require immediate Fleet Redeployment to handle permanent Volatility/Level shifts.")
print("3. Volatility Buffers recommended: Maintain 5% reserve fleet capacity specifically on City corridors to handle newly modeled erratic demand.")

Loading data...
Quantifying Stage 1 vs Q3 Actual Deviations...
Recalibrating Q4 Forecasts based on Shift Mechanisms...
Reassessing Operational Constraints & Risks...
Rendering Final Stage 2 Outputs...



STRATEGIC SUMMARY:
1. Feeder rationalization is mandatory. They suffered Negative Level Shifts indicating Metro-substitution.
2. Express Corridors (X66, X28, X11) require immediate Fleet Redeployment to handle permanent Volatility/Level shifts.
3. Volatility Buffers recommended: Maintain 5% reserve fleet capacity specifically on City corridors to handle newly modeled erratic demand.
